In [4]:
"""
Personalized Dashboard - Milestone 3

Takes the structured output from ai_interpretation_engine.py and produces
an actual dashboard-style PDF report - the "personalized dashboards" task.
Same reportlab pattern as Milestone 2's reading_report.py, extended with
four new sections: Interpretation, Personality, Recommendations, Life Trends.
"""

import os
from datetime import datetime
from xml.sax.saxutils import escape
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.colors import HexColor


def _p(text, style):
    """Escape text before wrapping in a Paragraph - same fix as Milestone 2's
    reading_report.py, avoids reportlab silently dropping angle-bracket text."""
    return Paragraph(escape(str(text)).replace("\n", "<br/>"), style)


def save_dashboard_report(analysis, combined_reading=None, output_path="dashboard_report.pdf", user_question=None):
    """
    analysis: the dict returned by ai_interpretation_engine.generate_full_analysis()
    combined_reading: optional - the dict from reading_report.generate_combined_reading().
        When given, the report includes a "Source Data" section showing the raw
        palm findings and tarot cards the AI analysis was grounded in - so a
        reader can see both the facts and the AI's synthesis, not just the AI's
        output on its own.
    """
    doc = SimpleDocTemplate(output_path, pagesize=letter)
    styles = getSampleStyleSheet()
    body_style = ParagraphStyle("body", parent=styles["Normal"], spaceAfter=10, leading=16)
    section_style = ParagraphStyle("section", parent=styles["Heading2"], textColor=HexColor("#5B3A8E"), spaceBefore=14)

    story = []
    story.append(Paragraph("Personalized Reading Dashboard", styles["Title"]))
    story.append(_p(f"Generated: {datetime.now().isoformat(timespec='seconds')}", styles["Normal"]))
    if analysis.get("_source"):
        story.append(_p(f"Analysis source: {analysis['_source']}", styles["Normal"]))
    story.append(Spacer(1, 12))

    # --- Source Data (the grounding facts, shown before the AI's synthesis) ---
    if combined_reading:
        story.append(Paragraph("Source Data", section_style))
        if combined_reading.get("tarot_question"):
            story.append(_p(f"Question asked: {combined_reading['tarot_question']}", body_style))

        palm_text = combined_reading.get("palm_text")
        if palm_text:
            for line_name, info in palm_text.items():
                label = line_name.replace("_", " ").title()
                story.append(_p(f"{label}: {info.get('finding', 'Not available.')}", body_style))
        elif combined_reading.get("palm_success"):
            story.append(_p("Palm analysis succeeded, but detailed line text was not captured for this session.", body_style))
        else:
            story.append(_p(f"Palm analysis unavailable: {combined_reading.get('palm_error', 'unknown reason')}", body_style))

        for card in combined_reading.get("cards_drawn", []):
            story.append(_p(f"{card['card_name']} ({card['orientation']}) - {card['meaning']}", body_style))

        story.append(Spacer(1, 8))

    # --- Interpretation ---
    story.append(Paragraph("Interpretation", section_style))
    story.append(_p(analysis.get("interpretation", "Not available."), body_style))

    # --- Personality Intelligence ---
    story.append(Paragraph("Personality Intelligence", section_style))
    personality = analysis.get("personality", {})
    strengths = personality.get("strengths", [])
    weaknesses = personality.get("weaknesses", [])
    story.append(_p(f"Strengths: {', '.join(strengths) if strengths else 'Not available.'}", body_style))
    story.append(_p(f"Weaknesses: {', '.join(weaknesses) if weaknesses else 'Not available.'}", body_style))
    story.append(_p(personality.get("behavioral_insights", "Not available."), body_style))

    # --- Recommendations ---
    story.append(Paragraph("Recommendations", section_style))
    recs = analysis.get("recommendations", {})
    story.append(_p(f"Personal growth: {recs.get('personal_growth', 'Not available.')}", body_style))
    story.append(_p(f"Relationships: {recs.get('relationships', 'Not available.')}", body_style))
    story.append(_p(f"Career: {recs.get('career', 'Not available.')}", body_style))

    # --- Life Trend Analysis ---
    story.append(Paragraph("Life Trend Analysis", section_style))
    trends = analysis.get("life_trends", {})
    story.append(_p(f"Opportunities: {trends.get('opportunities', 'Not available.')}", body_style))
    story.append(_p(f"Challenges: {trends.get('challenges', 'Not available.')}", body_style))
    story.append(_p(f"Growth potential: {trends.get('growth_potential', 'Not available.')}", body_style))

    doc.build(story)
    return output_path


if __name__ == "__main__":
    # Self-test with mock analysis data - proves the PDF builds correctly
    # for both a normal LLM-shaped result AND an empty/fallback one.
    mock_analysis = {
        "_source": "llm",
        "interpretation": "This reading points to a season of new beginnings, tempered by caution.",
        "personality": {"strengths": ["curious", "resilient"], "weaknesses": ["impulsive"], "behavioral_insights": "Tends to leap before fully weighing options."},
        "recommendations": {"personal_growth": "Slow down before big decisions.", "relationships": "Be more open with people close to you.", "career": "A new opportunity may be worth exploring."},
        "life_trends": {"opportunities": "A shift in direction is likely.", "challenges": "Avoid rushing key decisions.", "growth_potential": "High, if paired with patience."}
    }
    path = save_dashboard_report(
    analysis=mock_analysis,
    output_path="test_dashboard.pdf"
)
    print("Dashboard saved to:", path)
    print("File exists:", os.path.exists(path))

    # Also test the fallback shape (what happens if the LLM call failed)
    fallback_analysis = {
        "_source": "fallback (LLM call failed: test)",
        "interpretation": "AI analysis unavailable for this session.",
        "personality": {"strengths": [], "weaknesses": [], "behavioral_insights": "Not available."},
        "recommendations": {"personal_growth": "Not available.", "relationships": "Not available.", "career": "Not available."},
        "life_trends": {"opportunities": "Not available.", "challenges": "Not available.", "growth_potential": "Not available."}
    }
    path2 = save_dashboard_report(
    fallback_analysis,
    output_path="test_dashboard_fallback.pdf"
)
    print("Fallback dashboard saved to:", path2)
    print("File exists:", os.path.exists(path2))


Dashboard saved to: test_dashboard.pdf
File exists: True
Fallback dashboard saved to: test_dashboard_fallback.pdf
File exists: True
